# Day 25: Explainability Validation

YBI Foundation — Python AI & Data Science Internship

**Project:** AI-Based Student Performance Prediction System

## Objective

Verify explanation reproducibility, conservation and uncertainty across shuffle seeds.

## Theory

Permutation rankings vary with the random shuffle. Repeating a seed should reproduce its results. Impurity importance should conserve the total attribution when one-hot columns are grouped back into raw features.

## Prerequisites

Install `requirements-dev.txt` (and `requirements-api.txt` for API work) in the active Python environment. Use the repository checkout and the saved Day 17 model. The setup below locates or clones the repository for Colab. Run cells in order.


In [1]:
from pathlib import Path
import os, sys, subprocess, json
root = Path.cwd()
if not (root / "src").exists() and (root.parent / "src").exists():
    root = root.parent
if not (root / "src").exists():
    target = root / "IYB_INTERNSHIP"
    if not target.exists():
        subprocess.run(["git", "clone", "https://github.com/lightcode01-oss/IYB_INTERNSHIP.git", str(target)], check=True)
    root = target
os.chdir(root)
sys.path.insert(0, str(root))
print("Project:", root.name)


Project: IYB_INTERNSHIP


Load the verified pipeline and historical evaluation split.

In [2]:
from src.explainability import *
import numpy as np
import pandas as pd
pipeline=load_explainability_pipeline()
X,y=load_reference_dataset()
print("Reference rows:",len(X),"Features:",X.shape[1])

Reference rows: 130 Features: 30


Check reproducibility and aggregate feature importance.

In [3]:
a=compute_permutation_importance(pipeline,X,y,n_repeats=5,random_state=42)
b=compute_permutation_importance(pipeline,X,y,n_repeats=5,random_state=42)
pd.testing.assert_frame_equal(a,b)
trans,native=compute_native_feature_importance(pipeline)
assert np.isclose(trans.native_importance.sum(),native.native_importance.sum())
assert np.isclose(native.native_importance.sum(),1)
assert set(a.feature)==set(X.columns)
print("Same-seed reproducibility and attribution conservation: PASS")

[2026-11-02 14:03:34,106] INFO in model_explainability: Computing permutation importance with 5 repeats (scoring='f1_macro', seed=42)...


[2026-11-02 14:03:35,342] INFO in model_explainability: Computing permutation importance with 5 repeats (scoring='f1_macro', seed=42)...


Same-seed reproducibility and attribution conservation: PASS


Measure seed variability without selecting a new model.

In [4]:
runs=[]
for seed in [7,42,99]:
    r=compute_permutation_importance(pipeline,X,y,n_repeats=10,random_state=seed)
    r["seed"]=seed
    runs.append(r)
combined=pd.concat(runs,ignore_index=True)
summary=combined.groupby("feature").agg(mean_drop=("importance_mean","mean"),minimum_drop=("importance_mean","min"),maximum_drop=("importance_mean","max"),best_rank=("rank","min"),worst_rank=("rank","max")).sort_values("mean_drop",ascending=False)
summary.to_csv("reports/explainability_validation.csv")
print(summary.head(10).to_string())

[2026-11-02 14:03:37,262] INFO in model_explainability: Computing permutation importance with 10 repeats (scoring='f1_macro', seed=7)...


[2026-11-02 14:03:41,272] INFO in model_explainability: Computing permutation importance with 10 repeats (scoring='f1_macro', seed=42)...


[2026-11-02 14:03:44,734] INFO in model_explainability: Computing permutation importance with 10 repeats (scoring='f1_macro', seed=99)...


           mean_drop  minimum_drop  maximum_drop  best_rank  worst_rank
feature                                                                
failures    0.067904      0.060077      0.074229          1           1
Mjob        0.038872      0.035211      0.043115          2           2
freetime    0.023437      0.012058      0.032068          3           9
romantic    0.020265      0.015831      0.024692          4           6
absences    0.017569      0.014361      0.022237          3           7
famrel      0.016433      0.014245      0.018970          6           8
studytime   0.015956      0.002600      0.026672          3          14
school      0.013340      0.003093      0.020785          5          13
Fjob        0.012166      0.009777      0.014683          7          11
Pstatus     0.010701      0.010270      0.011490          8          11


## Findings and interpretation

The output identifies the leading variables and their seed sensitivity. Negative importance means shuffling improved this historical sample score; it is not evidence that a feature harms students.

## Limitations

Three seeds and ten shuffles quantify algorithmic variability, not population uncertainty. The test set was previously inspected. Importance does not imply causation.

## Development log

Implemented and ran the cells above during the repair and extension session on 2 November 2026. The saved outputs are produced by the kernel.

## Conclusion

Verify explanation reproducibility, conservation and uncertainty across shuffle seeds.

## Next-day plan

Probe model robustness under valid input changes and ordering variations.